# NB03 — Phase 3a: does the advantage survive rotation?

The key experiment.

**We plot three curves** 
We plot "R-id", "R-Q" and Qwen3-4B explainer curves so the claim becomes "R-Q falls to where
an unrelated 4B model sits".

**The explainer is initialized from unrotated `M`.** 
Because `M_Q ≡ M` as a function, feeding an unrotated-`M` explainer activations from `M_Q` holds behavioral similarity
identically fixed while destroying coordinate correspondence. We do not build an `M_Q` explainer.
This does not measure the self-explainer advantage, but rather the contribtion of the coordinate correspondence to that advantage.

| Hypothesis | Predicts for `E(from M)` reading `M_Q` |
|---|---|
| Behavioral self-simulation | advantage survives — behavior is bit-identical |
| Basis compatibility | advantage collapses toward the cross-model baseline |

**Arms** 
The core audit (NB00 sectoin 2) changed what this notebook has to run:

- `C0` under both rotations is the paper's own configuration. Each of the four `act_patch`
  configs do not build a projector. 
  They inject the target activation raw with a rank-128 LoRA downstream.
  This cannot represent `Q^{-1}` due to low rank, so we expect collapse due to capacity rather than self hood.
  We do this to test the exact set-up of the paper.
- `E_cross · P-rand` is the paper's cross-model condition. 
  We use a random dense projector with a rank-128 update (`C128`), because `model/utils.py:252–253` (on their repo) makes every trainable projector a LoRA target module. 
  A full-rank random projector is a different arm — `P-rand-full`, not to be conflated.
- `Cfull` remains the primary arm, and it is an augmentation we introduce so the question is
  answerable at all.

**We report everything normalized to the no-activation floor.** 
Rotation only destroys the coordinate frame information of `v`, but it has no effect on the information the explainer gains from the prompt itself. 
The paper's own `– activation` ablation bounds that value at ~4 points.
1 below measures our own floor at every `N` before any further runs.


In [1]:
# --- dependencies -----------------------------------------------------------
# A RunPod image ships torch built against the pod's own driver, so nothing here may
# replace it: se_env pins the installed torch as a pip constraint and installs only what
# is missing or below the floor these notebooks need — including bitsandbytes, which Colab
# had preinstalled and RunPod images do not (se_common asks for paged_adamw_8bit).
# Safe to re-run: a no-op on a warm pod.
import os
import sys

# `se/` holds the shared modules: se_env, se_config, rotate, se_common. Clone this repo
# onto the pod's volume (/workspace/self_explainer) so it survives the pod.
REPO_DIR = os.environ.get("SE_REPO_DIR", "/workspace/self_explainer")
if not os.path.isdir(os.path.join(REPO_DIR, "se")):
    REPO_DIR = os.path.abspath(".." if os.path.isdir("../se") else ".")
sys.path.insert(0, os.path.join(REPO_DIR, "se"))

import se_env

se_env.ensure_deps()


installing: transformers>=4.56.0 trl>=0.20.0 peft>=0.14.0 datasets>=3.0.0 accelerate>=1.0.0 bitsandbytes>=0.43.0 safetensors>=0.4.0 scikit-learn>=1.3.0 pandas>=2.0.0 matplotlib>=3.7.0 hf_transfer>=0.1.6
    transformers: absent -> >= 4.56.0
             trl: absent -> >= 0.20.0
            peft: absent -> >= 0.14.0
        datasets: absent -> >= 3.0.0
      accelerate: absent -> >= 1.0.0
    bitsandbytes: absent -> >= 0.43.0
     safetensors: absent -> >= 0.4.0
    scikit-learn: absent -> >= 1.3.0
          pandas: absent -> >= 2.0.0
      matplotlib: absent -> >= 3.7.0
     hf_transfer: absent -> >= 0.1.6
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 103.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 501.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 685.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 269.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 333.2 MB/s  0:00:00
 

{'transformers': '5.15.1',
 'trl': '1.10.0',
 'peft': '0.20.0',
 'datasets': '5.0.1',
 'accelerate': '1.14.0',
 'bitsandbytes': '0.50.1',
 'safetensors': '0.8.0',
 'scikit-learn': '1.9.0',
 'pandas': '3.0.5',
 'matplotlib': '3.11.1',
 'hf_transfer': '0.1.9'}

In [2]:
# --- environment ------------------------------------------------------------
# Caches and outputs go on the pod's volume, never the container disk: /workspace is what
# survives a stopped or terminated pod, and the 8B checkpoint alone is 16 GB. HF_TOKEN comes
# from the pod template's environment or <volume>/.hf_token — there is no prompt to answer,
# because a preempted pod restarts with nobody watching.
#
# Full sweep, three arms, LoRA + a full-rank input map on an 8B explainer.
env = se_env.bootstrap(repo_dir=REPO_DIR, gpu="required", min_vram_gb=80)

import se_config as C


RUNPOD ENVIRONMENT
host    : RunPod pod xx6cyd3zbg973t  |  python 3.12.3
gpu     : 1 x NVIDIA RTX PRO 6000 Blackwell Server Edition  |  95 GiB  |  bf16 yes
torch   : 2.8.0+cu128  (CUDA 12.8)
stack   : transformers 5.15.1 · peft 0.20.0 · trl 1.10.0 · datasets 5.0.1 · accelerate 1.14.0 · bitsandbytes 0.50.1
repo    : /workspace/self_explainer
volume  : /workspace  (own mount, 350836 GiB free)
hf cache: /workspace/.cache/huggingface/
outputs : /workspace/self_explainer
hf token: none
!! no HF token found. Public checkpoints still work; set HF_TOKEN in the pod template or write it to /workspace/.hf_token if a download 401s.


In [3]:
import json
import math
import time

import pandas as pd
import torch

import se_common as S

tokenizer = S.load_tokenizer()

datasets = {arm: S.load_ready_dataset(arm) for arm in ("identity", "Q")}
for arm, ds in datasets.items():
    print(f"{arm:>10}: {len(ds)} rows")

# matched data ordering across arms is a precondition, not a hope (v2 §3)
assert all(datasets["identity"][i]["messages"] == datasets["Q"][i]["messages"]
           for i in range(200)), "arms disagree on prompts — rebuild in NB02"
print("\narms share prompts, ordering, and labels; only the vectors differ")

# the explainer is loaded from the unrotated checkpoint — never from M_Q (v2 §2.1)
print(f"\nexplainer  : {C.EXPLAINER_MODEL_ID} (unrotated)")
print(f"N sweep    : {C.N_TRAIN_VALUES}")

# Every N runs every seed (supersedes v2 §3's "3 seeds at N in {512, 8192}"). Read out of
# C.seeds_for rather than restated, so this line cannot drift from the config the way it did
# when MULTI_SEED_N changed underneath it.
seed_grid = {n: C.seeds_for(n) for n in C.N_TRAIN_VALUES}
cut = [n for n, s in seed_grid.items() if len(s) < len(C.SEEDS)]
print(f"seeds      : {C.SEEDS} at every N — "
      f"{sum(len(s) for s in seed_grid.values())} (N x seed) cells per full-sweep arm")
if cut:
    print(f"             CUT to a single seed at N in {cut}; the seed band does not exist "
          f"there, and §9's 'overlap' vs 'separated' readings are undecidable at those N")
print(f"eval set   : {C.EVAL_SIZE} held-out examples "
      f"(8x the original; see NB00 §6 for why)")


  identity: 20000 rows
         Q: 20000 rows

arms share prompts, ordering, and labels; only the vectors differ

explainer  : Qwen/Qwen3-8B (unrotated)
N sweep    : [256, 512, 1024, 2048, 4096, 8192, 16384]
seeds      : [67, 68, 69] at every N — 21 (N x seed) cells per full-sweep arm
eval set   : 1024 held-out examples (8x the original; see NB00 §6 for why)


## 1. The no-activation floor — the scale every other number is on

Rotation will only destroy the signal from `v` and will not effect what the explainer gets from
the prompt itself. 
The paper bounds that value at 64.0 → 59.9 for Qwen3-8B self-explanation (Table 5).
So the entire measurable range of the rotation arm is only about 4 points, and we report all results normalized to this floor.

We need to rerun this to get our own floor because we are using different `N` and different eval-sets.
We set `v = 0` and the prompt, template, label distribution, the training budget all stay identical.


In [4]:
FLOOR_N = C.N_TRAIN_VALUES     # cut to C.N_TRAIN_LADDER first if the budget bites

# v = 0 keeps prompts, labels, ordering and training budget identical: the only thing removed
# is the information carried by the activation.
zeroed = datasets["identity"].map(lambda ex: {
    "patch_position": {**ex["patch_position"],
                       "intervention_vector": [0.0] * len(
                           ex["patch_position"]["intervention_vector"])},
})

# Every seed, like every other arm — and here for one extra reason. The floor is the
# DENOMINATOR of `retained`, and `retained` is a ratio with a ~4-point denominator, so the
# floor's noise propagates into every normalized number more strongly than the numerator's
# does (prereg_threshold_justification.md §1.1: SE(retained) ~ 0.30 at retained = 0, rising
# with retained). Run single-seed, the denominator of every result in this project would be
# its least-replicated quantity.
# `seeds_for_arm`, not `seeds_for`: the floor is (Cfull, zerovec) and was never replicated
# at 16,384, so the plain grid would ask for two new floor runs there — and moving the floor
# at that N would move every normalized number at that N.
FLOOR_CELLS = [dict(n_train=n, rotation="zerovec", capacity="Cfull", init="identity",
                    seed=seed)
               for n in FLOOR_N for seed in C.seeds_for_arm(n, "Cfull", "zerovec")]
PLAN = S.plan_runs(FLOOR_CELLS, label="NB03 §1 no-activation floor")
TRAIN_NEW = True          # set False to price this cell without starting anything

floor_results = []
for n in FLOOR_N:
    for seed in C.seeds_for_arm(n, "Cfull", "zerovec"):
        if not TRAIN_NEW and not S.run_is_done(n, "zerovec", "Cfull", "identity", seed=seed):
            continue
        t0 = time.time()
        print(f"\n=== no-activation floor (v = 0) · n={n} · seed={seed} " + "=" * 14)
        kw = dict(tokenizer=tokenizer, dataset=zeroed, seed=seed)
        S.run_training(n, "zerovec", "Cfull", "identity", **kw)
        scores = S.eval_run(n, "zerovec", "Cfull", "identity", **kw)
        scores["label"] = "no-activation floor (v=0)"
        floor_results.append(scores)
        print(f"  exact_match {scores['exact_match']:.3f} | "
              f"content_match {scores['content_match']:.3f} [{(time.time()-t0)/60:.1f} min]")

METRIC_COLS = ["exact_match", "has_changed_f1", "content_match"]

# One row per (N, seed) is kept so NB07 can put a band on the denominator; `floor_df` is the
# per-N mean, which is what everything downstream divides by.
floor_runs = pd.DataFrame(floor_results)
floor_runs.to_csv(f"{C.REPORTS_DIR}/no_activation_floor_runs.csv", index=False)

floor_df = floor_runs.groupby("n_train", as_index=False)[METRIC_COLS].mean()
floor_spread = (floor_runs.groupby("n_train", as_index=False)[METRIC_COLS]
                .agg(lambda x: x.max() - x.min()))
floor_df.to_csv(f"{C.REPORTS_DIR}/no_activation_floor.csv", index=False)
floor_spread.to_csv(f"{C.REPORTS_DIR}/no_activation_floor_spread.csv", index=False)
FLOOR = {int(r["n_train"]): r for _, r in floor_df.iterrows()}


def floor_at(n, metric="exact_match"):
    """The measured floor at N, falling back to the nearest measured N in log space."""
    n = int(n)
    if n in FLOOR:
        return FLOOR[n][metric]
    nearest = min(FLOOR, key=lambda k: abs(math.log2(k) - math.log2(n)))
    return FLOOR[nearest][metric]


untrained = S.eval_run(None, "identity", "Cfull", "identity", tokenizer,
                       dataset=datasets["identity"])

print("\nNO-ACTIVATION FLOOR (mean over seeds)")
print("=" * 62)
print(floor_df[["n_train"] + METRIC_COLS].round(3).to_string(index=False))
print("\nseed spread at each N (max - min) — this is the denominator's own uncertainty,")
print("and it enters every `retained` figure through the ratio:")
print(floor_spread[["n_train"] + METRIC_COLS].round(3).to_string(index=False))
print(f"\nuntrained explainer (no adapter at all): "
      f"exact_match {untrained['exact_match']:.3f}")
print(f"paper's Table 5 floor for comparison   : {C.PAPER_NO_ACTIVATION:.3f} "
      f"(their N, their eval set — not used in any reading)")
print("\nEvery arm below is reported twice: raw, and as the fraction of the gap between this")
print("floor and the unrotated reference that it retains. The second is the readable one.")


PLAN — NB03 §1 no-activation floor
  19 already on disk (resume will skip), 0 to train
  nothing to train — this cell is a re-read

=== no-activation floor (v = 0) · n=256 · seed=67 ==============
[skip] /workspace/self_explainer/runs/patching/expl_Qwen3-8B/rot_zerovec/cap_Cfull/init_identity/n_train_256 already done
  exact_match 0.332 | content_match 0.545 [0.0 min]

=== no-activation floor (v = 0) · n=256 · seed=68 ==============
[skip] /workspace/self_explainer/runs/patching/expl_Qwen3-8B/rot_zerovec/cap_Cfull/init_identity/n_train_256/seed_68 already done
  exact_match 0.329 | content_match 0.553 [0.0 min]

=== no-activation floor (v = 0) · n=256 · seed=69 ==============
[skip] /workspace/self_explainer/runs/patching/expl_Qwen3-8B/rot_zerovec/cap_Cfull/init_identity/n_train_256/seed_69 already done
  exact_match 0.302 | content_match 0.539 [0.0 min]

=== no-activation floor (v = 0) · n=512 · seed=67 ==============
[skip] /workspace/self_explainer/runs/patching/expl_Qwen3-8B/rot_ze

## 2. The sweep

We have a four arm sweep.
The two full-sweep arms are the ones the figure turns on while `C0` runs at `N_TRAIN_LADDER` only.

| Arm | Whose | What it is |
|---|---|---|
| `E_self · Cfull · R-id` | ours | the augmented self-explainer, unrotated |
| `E_self · Cfull · R-Q` | ours | the same, reading a rotated frame |
| `E_self · C0 · R-id` | **the paper's** | no map at all — the act_patch configuration (F.1) |
| `E_self · C0 · R-Q` | **the paper's** | the same, rotated: cannot represent `Q^{-1}` |

Each cell writes its own directory and is skipped if complete, so this is resumable across pod
restarts. It is ordered by `N` and then by arm, so if the pod dies halfway we have every arm at small `N` rather
than one arm everywhere.

Two full-sweep arms across seven `N` at three seeds (42), plus `C0` at two `N` under two rotations at three seeds (12), plus §1's floor at seven `N` at three seeds (21), plus §3's init control (4).


In [5]:
ARMS = [
    # (label,                  rotation,   capacity,  init,       explainer,            N values)
    ("E_self · Cfull · R-id", "identity", "Cfull",   "identity", C.EXPLAINER_MODEL_ID, C.N_TRAIN_VALUES),
    ("E_self · Cfull · R-Q",  "Q",        "Cfull",   "identity", C.EXPLAINER_MODEL_ID, C.N_TRAIN_VALUES),
    # the paper's own configuration, under both rotations (revised §7.1)
    ("E_self · C0 · R-id",    "identity", "C0",      "identity", C.EXPLAINER_MODEL_ID, C.N_TRAIN_LADDER),
    ("E_self · C0 · R-Q",     "Q",        "C0",      "identity", C.EXPLAINER_MODEL_ID, C.N_TRAIN_LADDER),
]

# Seeds come from `seeds_for_arm`, which knows that only Cfull was replicated at 16,384.
# Under plain `seeds_for` this loop asks for C0 x {R-id, R-Q} x {68, 69} there — four runs,
# ~6 h — and nothing reads C0 at that N as a 3-seed quantity, because C8/C128/C512/Oracle
# are single-seed there too (see C.SEED_REPLICATED_AT_MAX_N).
SWEEP_CELLS = [dict(n_train=n, rotation=rot, capacity=cap, init=init,
                    explainer_model_id=expl, seed=seed)
               for n in C.N_TRAIN_VALUES
               for label, rot, cap, init, expl, n_values in ARMS if n in n_values
               for seed in C.seeds_for_arm(n, cap, rot)]
PLAN = S.plan_runs(SWEEP_CELLS, label="NB03 §2 core sweep")
TRAIN_NEW = True          # set False to price this cell without starting anything

results = []
for n in C.N_TRAIN_VALUES:
    for label, rot, cap, init, expl, n_values in ARMS:
        if n not in n_values:
            continue
        for seed in C.seeds_for_arm(n, cap, rot):
            if not TRAIN_NEW and not S.run_is_done(n, rot, cap, init, seed=seed,
                                                   explainer_model_id=expl):
                continue
            t0 = time.time()
            print(f"\n=== {label} · n={n} · seed={seed} " + "=" * 24)
            kw = dict(tokenizer=tokenizer, dataset=datasets[rot],
                      explainer_model_id=expl, seed=seed)
            S.run_training(n, rot, cap, init, **kw)
            scores = S.eval_run(n, rot, cap, init, **kw)
            scores["label"] = label
            results.append(scores)
            print(f"  exact_match {scores['exact_match']:.3f} | "
                  f"has_changed_f1 {scores['has_changed_f1']:.3f} | "
                  f"content_match {scores['content_match']:.3f} "
                  f"[{(time.time()-t0)/60:.1f} min]")

core = pd.DataFrame(results)
core.to_csv(f"{C.REPORTS_DIR}/core_sweep.csv", index=False)
core.groupby(["label", "n_train"]).exact_match.agg(["mean", "min", "max", "count"]).round(3)


PLAN — NB03 §2 core sweep
  56 already on disk (resume will skip), 0 to train
  nothing to train — this cell is a re-read

=== E_self · Cfull · R-id · n=256 · seed=67 ========================
[skip] /workspace/self_explainer/runs/patching/expl_Qwen3-8B/rot_identity/cap_Cfull/init_identity/n_train_256 already done
  exact_match 0.335 | has_changed_f1 0.500 | content_match 0.552 [0.0 min]

=== E_self · Cfull · R-id · n=256 · seed=68 ========================
[skip] /workspace/self_explainer/runs/patching/expl_Qwen3-8B/rot_identity/cap_Cfull/init_identity/n_train_256/seed_68 already done
  exact_match 0.323 | has_changed_f1 0.525 | content_match 0.558 [0.0 min]

=== E_self · Cfull · R-id · n=256 · seed=69 ========================
[skip] /workspace/self_explainer/runs/patching/expl_Qwen3-8B/rot_identity/cap_Cfull/init_identity/n_train_256/seed_69 already done
  exact_match 0.299 | has_changed_f1 0.520 | content_match 0.520 [0.0 min]

=== E_self · Cfull · R-Q · n=256 · seed=67 ==============

mean    min    max  count
label                 n_train                            
E_self · C0 · R-Q     512      0.383  0.345  0.411      3
                      2048     0.516  0.514  0.518      3
                      16384    0.706  0.706  0.706      1
E_self · C0 · R-id    512      0.380  0.357  0.396      3
                      2048     0.496  0.483  0.513      3
                      16384    0.701  0.701  0.701      1
E_self · Cfull · R-Q  256      0.348  0.327  0.378      3
                      512      0.435  0.409  0.460      3
                      1024     0.481  0.460  0.497      3
                      2048     0.536  0.526  0.547      3
                      4096     0.600  0.582  0.621      3
                      8192     0.654  0.644  0.664      3
                      16384    0.729  0.725  0.734      3
E_self · Cfull · R-id 256      0.319  0.299  0.335      3
                      512      0.418  0.411  0.428      3
                      1024     0.469  0.440  0.485      3
                      2048     0.519  0.510  0.525      3
                      4096     0.593  0.580  0.607      3
                      8192     0.660  0.651  0.666      3
                      16384    0.717  0.708  0.724      3

In [6]:
for seed in (68, 69):
      for rot in ("identity", "Q"):
          kw = dict(tokenizer=tokenizer, dataset=datasets[rot], seed=seed)
          S.run_training(16384, rot, "Cfull", "identity", **kw)
          print(S.eval_run(16384, rot, "Cfull", "identity", **kw))

[skip] /workspace/self_explainer/runs/patching/expl_Qwen3-8B/rot_identity/cap_Cfull/init_identity/n_train_16384/seed_68 already done
{'n_train': 16384, 'rotation': 'identity', 'capacity': 'Cfull', 'init': 'identity', 'task': 'patching', 'explainer': 'Qwen/Qwen3-8B', 'seed': 68, 'exact_match': 0.7197265625, 'has_changed_f1': 0.8144354281833823, 'content_match': 0.7783203125, 'unparseable_rate': 0.0}
[skip] /workspace/self_explainer/runs/patching/expl_Qwen3-8B/rot_Q/cap_Cfull/init_identity/n_train_16384/seed_68 already done
{'n_train': 16384, 'rotation': 'Q', 'capacity': 'Cfull', 'init': 'identity', 'task': 'patching', 'explainer': 'Qwen/Qwen3-8B', 'seed': 68, 'exact_match': 0.7294921875, 'has_changed_f1': 0.8261393769434746, 'content_match': 0.7958984375, 'unparseable_rate': 0.0}
[skip] /workspace/self_explainer/runs/patching/expl_Qwen3-8B/rot_identity/cap_Cfull/init_identity/n_train_16384/seed_69 already done
{'n_train': 16384, 'rotation': 'identity', 'capacity': 'Cfull', 'init': 'iden

## 3. The init-head-start control

`Cfull` under R-id is initialized at identity (at the solution) whereas `Cfull` under R-Q is initialized at identity when the truth is `Q^T`. 
So if R-id separates from R-Q at low `N` and converges at high `N` then this is a consequence of initialization and not self-explanation.

`Cfull-rand` starts both rotation arms at a random orthogonal map, putting them the same distance from their respective solutions.
Two `N` values is enough to bound it.

In [7]:
# filtering_retrain_decision.md: "Run seeds 68 and 69 at N = 512 and 2,048 only ... Skip the
# extra Cfull-rand seeds at N=16,384 — the two inits already corroborate each other there."
# `seeds_for_arm` enforces exactly that; plain `seeds_for` would add four runs at 16,384.
RAND_CELLS = [dict(n_train=n, rotation=rot, capacity="Cfull-rand", init="orthogonal",
                   seed=seed)
              for n in C.N_TRAIN_LADDER
              for rot in ("identity", "Q")
              for seed in C.seeds_for_arm(n, "Cfull-rand", rot)]
PLAN = S.plan_runs(RAND_CELLS, label="NB03 §3 Cfull-rand init control")
TRAIN_NEW = True          # set False to price this cell without starting anything

rand_results = []
for n in C.N_TRAIN_LADDER:
    for rot in ("identity", "Q"):
        for seed in C.seeds_for_arm(n, "Cfull-rand", rot):
            if not TRAIN_NEW and not S.run_is_done(n, rot, "Cfull-rand", "orthogonal",
                                                   seed=seed):
                continue
            t0 = time.time()
            print(f"\n=== Cfull-rand · {C.ROTATION_LABEL[rot]} · n={n} · seed={seed} "
                + "=" * 18)
            kw = dict(tokenizer=tokenizer, dataset=datasets[rot], seed=seed)
            S.run_training(n, rot, "Cfull-rand", "orthogonal", **kw)
            scores = S.eval_run(n, rot, "Cfull-rand", "orthogonal", **kw)
            scores["label"] = f"E_self · Cfull-rand · {C.ROTATION_LABEL[rot]}"
            rand_results.append(scores)
            print(f"  exact_match {scores['exact_match']:.3f} | "
                f"has_changed_f1 {scores['has_changed_f1']:.3f} | "
                f"content_match {scores['content_match']:.3f} "
                f"[{(time.time()-t0)/60:.1f} min]")

if rand_results:
    rd = pd.DataFrame(rand_results)
    rd.to_csv(f"{C.REPORTS_DIR}/cfull_rand_runs.csv", index=False)
    print()
    print(rd.pivot_table(index="n_train", columns="rotation", values="exact_match",
                       aggfunc=["mean", "min", "max"]).round(3).to_string())
    print("\nA gap that survives here is not an initialization artifact.")


PLAN — NB03 §3 Cfull-rand init control
  14 already on disk (resume will skip), 0 to train
  nothing to train — this cell is a re-read

=== Cfull-rand · R-id · n=512 · seed=67 ==================
[skip] /workspace/self_explainer/runs/patching/expl_Qwen3-8B/rot_identity/cap_Cfull-rand/init_orthogonal/n_train_512 already done
  exact_match 0.417 | has_changed_f1 0.607 | content_match 0.593 [0.0 min]

=== Cfull-rand · R-id · n=512 · seed=68 ==================
[skip] /workspace/self_explainer/runs/patching/expl_Qwen3-8B/rot_identity/cap_Cfull-rand/init_orthogonal/n_train_512/seed_68 already done
  exact_match 0.421 | has_changed_f1 0.641 | content_match 0.611 [0.0 min]

=== Cfull-rand · R-id · n=512 · seed=69 ==================
[skip] /workspace/self_explainer/runs/patching/expl_Qwen3-8B/rot_identity/cap_Cfull-rand/init_orthogonal/n_train_512/seed_69 already done
  exact_match 0.423 | has_changed_f1 0.644 | content_match 0.638 [0.0 min]

=== Cfull-rand · R-Q · n=512 · seed=67 ==============

## 4. Normalized to the floor, which is where the readings live

`retained = (score − floor) / (score(Cfull, R-id) − floor)` at matched `N`:

- `1.0` — the arm keeps everything the activation was contributing
- `0.0` — the arm is at the no-activation floor; rotation destroyed all of it
- `< 0` — worse than passing no activation at all, which is a bug or an actively misleading
  vector, and §9 says investigate before reporting

`destroyed = 1 − retained` is what the preregistered thresholds are stated in (`overlap_destroyed = 0.15`, `separated_destroyed = 0.35`). 
The raw scale is printed alongside, because a reader comparing against the paper's tables needs it.

**We use a paired difference** 
Every arm is scored on the same held-out examples, so we need paired difference.
McNemar's discordant counts give `se = √(b + c) / n`, several times tighter than `√2 ×` the per-arm half-width.


In [8]:
METRIC = "exact_match"

ref_by_n = (core[core.label == "E_self · Cfull · R-id"]
            .groupby("n_train")[METRIC].mean().to_dict())

rows = []
for _, r in core.iterrows():
    n = int(r["n_train"])
    fl, ref = floor_at(n, METRIC), ref_by_n.get(n)
    rows.append({
        "label": r["label"], "n_train": n, "seed": r["seed"],
        "raw": r[METRIC], "floor": fl, "reference": ref,
        "span": C.CONTRIBUTION_SPAN,
        "retained": S.fraction_retained(r[METRIC], fl),
    })
norm = pd.DataFrame(rows)
norm["destroyed"] = 1 - norm["retained"]
norm.to_csv(f"{C.REPORTS_DIR}/core_sweep_normalized.csv", index=False)

print(f"RETAINED FRACTION of the activation's contribution ({METRIC})")
print(f"denominator: C.CONTRIBUTION_SPAN = {C.CONTRIBUTION_SPAN:.4f} (the paper's Table 5")
print(" `- activation` span). It is a CONSTANT -- 1.0 is the paper's whole activation")
print(" contribution, NOT our own R-id arm, which no longer sits at 1.0 by construction.")
print(" Its row below therefore reads as a measurement: how much of the paper's span our")
print(" unrotated arm opens at each N. See se_config.CONTRIBUTION_SPAN for why the per-N")
print(" `reference - floor` denominator was retired.")
print("=" * 78)
print(norm.pivot_table(index="n_train", columns="label", values="retained", aggfunc="mean")
      .round(3).to_string())
print("\nraw scores, same layout:")
print(norm.pivot_table(index="n_train", columns="label", values="raw", aggfunc="mean")
      .round(3).to_string())
print("\nfloor by N:")
print(floor_df.set_index("n_train")[METRIC].round(3).to_string())

# --- paired differences on the same eval items ------------------------------
print("\nPAIRED R-id vs R-Q at Cfull (same held-out examples, McNemar)")
print("=" * 78)
paired_rows = []
for n in C.N_TRAIN_VALUES:
    a = S.load_eval_records(n, "identity", "Cfull", "identity")
    b = S.load_eval_records(n, "Q", "Cfull", "identity")
    if not a or not b:
        continue
    d = S.paired_delta(a, b, key=METRIC)
    # Constant denominator, so `destroyed` is an exact rescale of the paired delta and its
    # CI rescales with it -- half-width below is 1.96*se/span, not a Fieller interval.
    span = C.CONTRIBUTION_SPAN
    d.update(n_train=n, span=span, destroyed=d["delta"] / span,
             destroyed_ci95=[(d["delta"] - 1.96 * d["se"]) / span,
                             (d["delta"] + 1.96 * d["se"]) / span])
    paired_rows.append(d)
    print(f"  N={n:>5}: delta {d['delta']:+.4f} +/- {1.96 * d['se']:.4f}  "
          f"(discordant {d['discordant_a']}/{d['discordant_b']})  "
          f"= {d['destroyed']:+.2f} [{d['destroyed_ci95'][0]:+.2f}, "
          f"{d['destroyed_ci95'][1]:+.2f}] of the activation's contribution")

if paired_rows:
    with open(f"{C.REPORTS_DIR}/paired_deltas.json", "w") as f:
        json.dump(paired_rows, f, indent=2, default=S.json_default)
    print("\nA paired CI that excludes 0 is the strongest statement this eval size supports.")
else:
    print("  (no eval_records.json found — they are written by eval_run alongside the scores)")


RETAINED FRACTION of the activation's contribution (exact_match)
denominator: C.CONTRIBUTION_SPAN = 0.0410 (the paper's Table 5
 `- activation` span). It is a CONSTANT -- 1.0 is the paper's whole activation
 contribution, NOT our own R-id arm, which no longer sits at 1.0 by construction.
 Its row below therefore reads as a measurement: how much of the paper's span our
 unrotated arm opens at each N. See se_config.CONTRIBUTION_SPAN for why the per-N
 `reference - floor` denominator was retired.
label    E_self · C0 · R-Q  E_self · C0 · R-id  E_self · Cfull · R-Q  E_self · Cfull · R-id
n_train                                                                                    
256                    NaN                 NaN                 0.651                 -0.048
512                  0.111               0.048                 1.374                  0.969
1024                   NaN                 NaN                 1.326                  1.024
2048                 0.326              -

## 5. The main figure

Raw scores on top with the floor too. 
Below, the fraction of the activations contribution.

The normalized panel is a row because the floor and the reference both move with `N`.


In [ ]:
import math

import matplotlib.pyplot as plt

METRICS = ["exact_match", "has_changed_f1", "content_match"]
COLORS = {"E_self · Cfull · R-id": "#1b6ca8",
          "E_self · Cfull · R-Q": "#c0392b",
          "E_self · C0 · R-id": "#2e86c1",
          "E_self · C0 · R-Q": "#e67e22"}
DASHED = {"E_self · C0 · R-id", "E_self · C0 · R-Q"}      # the paper's configuration

fig, axes = plt.subplots(2, 3, figsize=(16, 9.4), sharex=True)
for j, m in enumerate(METRICS):
    ax = axes[0][j]
    for label, color in COLORS.items():
        sub = core[core.label == label].groupby("n_train")[m].agg(["mean", "min", "max"])
        if not len(sub):
            continue
        sub = sub.sort_index()
        ax.plot(sub.index, sub["mean"], marker="o", color=color, label=label,
                linestyle="--" if label in DASHED else "-")
        ax.fill_between(sub.index, sub["min"], sub["max"], color=color, alpha=0.18)
    fl = floor_df.set_index("n_train")[m].sort_index()
    ax.plot(fl.index, fl.values, color="black", linestyle=":", linewidth=1.4,
            label="no-activation floor (v=0)")
    ax.set_xscale("log", base=2)
    ax.set_title(m.replace("_", " "))
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)

    # Bottom row: distance above the no-activation floor at that N, in the metric's OWN
    # points, for all three columns. No denominator anywhere in this row.
    #
    # Dropped 2026-08-27. C.CONTRIBUTION_SPAN is a CONSTANT, so dividing by it is an exact
    # linear rescale of `score - floor` (se_common.fraction_retained): it changes no
    # ordering, no sign, no CI width and no threshold decision — only the axis label. It is
    # also Table 5's `- activation` span for EXACT MATCH alone; the paper reports no such
    # ablation for has_changed_f1 or content_match, and the per-N `reference - floor` that
    # might stand in for one is the Fieller quantity retired on 2026-08-24
    # (se_config.CONTRIBUTION_SPAN). So normalizing one column of three bought nothing and
    # cost the row a shared scale. In points throughout, the three panels are directly
    # comparable and share a y-axis.
    #
    # Note the prereg thresholds (0.15 / 0.35 destroyed) are NOT drawable here in either
    # scale: they are thresholds on the GAP BETWEEN two arms, not on either arm's height
    # above the floor. The normalized reading still lives where it is used —
    # core_sweep_normalized.csv, the paired-delta CIs above, and NB07's readings table.
    axn = axes[1][j]
    for label, color in COLORS.items():
        sub = core[core.label == label]
        if not len(sub):
            continue
        y = [v - floor_at(n, m) for n, v in zip(sub["n_train"], sub[m])]
        g = (pd.Series(y, index=sub["n_train"])
             .groupby(level=0).agg(["mean", "min", "max"]).sort_index())
        axn.errorbar(g.index, g["mean"],
                     yerr=[g["mean"] - g["min"], g["max"] - g["mean"]],
                     marker="o", capsize=3, color=color,
                     linestyle="--" if label in DASHED else "-")
    axn.axhline(0, color="black", linestyle=":", linewidth=1.4)
    axn.set_xscale("log", base=2)
    axn.set_xlabel("N_TRAIN")
    axn.grid(alpha=0.3)
    axn.set_ylabel(f"{m.replace('_', ' ')}\npoints above the floor")
    if m == "exact_match":
        # The paper's whole activation contribution, drawn in the same points as the
        # curves rather than used as a divisor. This is the one thing the old normalized
        # axis carried that is worth keeping, and it survives the change unaltered.
        axn.axhline(C.CONTRIBUTION_SPAN, color="#7f8c8d", linestyle="--", linewidth=0.9)
        axn.set_title(f"grey line = the paper's activation contribution, "
                      f"{C.CONTRIBUTION_SPAN:.3f} (Table 5)", fontsize=9)
    else:
        axn.set_title("the paper reports no activation ablation "
                      "for this metric", fontsize=9)

# Pin the x-axis to the sweep grid. `errorbar` on the bottom row autoscales in LINEAR
# space before `set_xscale("log")` is applied, so its lower bound goes negative and the log
# transform clamps it to ~5 instead of the smallest N; with sharex=True that bogus limit
# then propagates to every panel and the axis ran from 2^2 instead of 2^8.
# One scale for the whole bottom row now that every panel is in points. Taken from the
# union of the three autoscaled ranges rather than hardcoded, so it tracks the data.
lo = min(a.get_ylim()[0] for a in axes[1])
hi = max(a.get_ylim()[1] for a in axes[1])
for a in axes[1]:
    a.set_ylim(lo, hi)

NS = sorted(int(n) for n in core["n_train"].dropna().unique())
for ax in axes.flat:
    ax.set_xlim(NS[0] / 1.3, NS[-1] * 1.3)
    ax.set_xticks(NS)
    ax.set_xticklabels([f"$2^{{{int(round(math.log2(n)))}}}$" for n in NS])
    ax.minorticks_off()

axes[0][0].set_ylabel("score (raw)")
axes[0][0].legend(fontsize=7.5)
fig.suptitle(
    "Rotation vs the paper's own C0 configuration. Bands = min/max over seeds "
    "(bottom row: computed per seed, not from the seed-mean).\n"
    "Bottom row is distance above the no-activation floor at that N, in each metric's own "
    "points \u2014 no normalization, so all three panels share one scale.\nGrey dashed line "
    f"(exact match only) is the paper's {C.CONTRIBUTION_SPAN:.3f} Table 5 activation "
    "contribution, for reference.")
fig.tight_layout()
fig.savefig(f"{C.FIGURES_DIR}/core_sweep.png", dpi=150)
plt.show()


In [10]:
# the gap, the band, and the normalized scale the readings are stated on
prereg = json.load(open(f"{C.REPORTS_DIR}/preregistration.json"))
primary = prereg["primary_metric"]
TH = prereg["thresholds"]

piv = core.pivot_table(index="n_train", columns="label", values=primary, aggfunc="mean")
piv["gap (R-id - R-Q)"] = (piv.get("E_self · Cfull · R-id", 0)
                           - piv.get("E_self · Cfull · R-Q", 0))
piv["floor"] = [floor_at(n, primary) for n in piv.index]
piv["retained (R-Q)"] = [
    S.fraction_retained(piv.loc[n].get("E_self · Cfull · R-Q", float("nan")),
                        floor_at(n, primary))
    for n in piv.index
]
piv["retained (R-id)"] = [
    S.fraction_retained(piv.loc[n].get("E_self · Cfull · R-id", float("nan")),
                        floor_at(n, primary))
    for n in piv.index
]
# `destroyed` is what rotation removed, so it is measured against the unrotated arm's own
# position on the fixed scale, not against 1.0 — R-id is no longer pinned there.
piv["destroyed"] = piv["retained (R-id)"] - piv["retained (R-Q)"]

spread = core.groupby(["label", "n_train"])[primary].agg(lambda x: x.max() - x.min())
band_width = spread[spread > 0]

print(f"{primary}: raw scores, the raw gap, and the normalized quantity\n")
print(piv.round(4).to_string())
print("\nseed band width (max - min), where >1 seed was run:")
print(band_width.round(4).to_string() if len(band_width) else "  (single seed everywhere)")
print(f"\nThresholds are on the NORMALIZED scale (revised §9): overlap if destroyed < "
      f"{TH['overlap_destroyed']}, separated if destroyed > {TH['separated_destroyed']} "
      f"and the seed bands do not overlap.")
print(f"Raw-scale equivalents (the same at every N, because the denominator is the "
      f"constant {C.CONTRIBUTION_SPAN:.4f}): overlap < "
      f"{TH['overlap_destroyed'] * C.CONTRIBUTION_SPAN:.4f}, separated > "
      f"{TH['separated_destroyed'] * C.CONTRIBUTION_SPAN:.4f} exact-match points — which is "
      f"why the raw thresholds from v1 of the preregistration were unreachable. These match "
      f"the raw-equivalent column of prereg_threshold_justification.md §2 exactly, which the "
      f"old per-N denominator did not.")


exact_match: raw scores, the raw gap, and the normalized quantity

label    E_self · C0 · R-Q  E_self · C0 · R-id  E_self · Cfull · R-Q  E_self · Cfull · R-id  gap (R-id - R-Q)   floor  retained (R-Q)  retained (R-id)  destroyed
n_train                                                                                                                                                          
256                    NaN                 NaN                0.3477                 0.3190           -0.0286  0.3210          0.6510          -0.0476    -0.6987
512                 0.3828              0.3802                0.4346                 0.4180           -0.0166  0.3783          1.3735           0.9686    -0.4049
1024                   NaN                 NaN                0.4811                 0.4688           -0.0124  0.4268          1.3259           1.0242    -0.3017
2048                0.5163              0.4961                0.5361                 0.5189           -0.0173  0.5029      

**Do not classify the outcome here.** NB07 applies the preregistered rules mechanically to
everything at once. Reading this table and deciding what it means is the step preregistration
exists to prevent.

**Exit criteria (revised §7.1).** One figure: `Cfull` under both rotations, `C0` under both, the
cross-model baseline at each `N`, seed bands throughout, and the second row showing the fraction
of the activation's contribution retained. If `C0` collapses and `Cfull` does not, that is a
capacity result about the paper's configuration — label it, and do not let it stand in for the
basis finding.

Next: **NB04** turns capacity from a pinned constant into the measurement, and runs the exactness
check that catches plumbing bugs for free.
